# Altima MMM — Constrained Marketing Mix Model (v2)

**Pipeline:** Load → VIF → OLS (Unconstrained) → Constrained MLR → Contribution → Export

**Key Findings:**
- OLS R² = 0.622 (unconstrained baseline)
- Constrained R² ≈ 0.30–0.41 (all business signs enforced)
- RAD_SUP & PRT_SUP: VIF = 202 (severe multicollinearity)
- Gas Price: positive correlation (+0.55) — conflicts with negative business sign

## 1. Libraries & Setup

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.optimize import minimize
from sklearn.metrics import mean_absolute_percentage_error
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

In [ ]:
data   = pd.read_excel('TRANSFORMED_DATA_INPUT_V1.xlsx')
contri = pd.read_excel('CONTRIBUTION_V1.xlsx')
data.columns = data.columns.str.strip()
contri.columns = contri.columns.str.strip()
data['Time_Period'] = pd.to_datetime(data['Time_Period'])

dependent_var    = 'ALT_DEP'
independent_vars = ['TVT_SUP','YTB_SUP','RAD_SUP','PRT_SUP','GAS_PRI','DIS_COU','SEA_ALT']
cols             = ['const'] + independent_vars

y      = data[dependent_var].values
X_raw  = data[independent_vars].values
X_orig = np.hstack([np.ones((len(y), 1)), X_raw])
n, p   = len(y), len(independent_vars)
print(f'Data: {data.shape}, n={n}, Total Actual={y.sum():,.0f}')

## 3. VIF — Multicollinearity

**Thresholds:** VIF < 5 = Low | 5–10 = Moderate | > 10 = SEVERE

⚠ RAD_SUP & PRT_SUP have VIF=202 — near-identical after APL transformation.

In [ ]:
def compute_vif(X_m, names, y_dep):
    rows = []
    for i,var in enumerate(names):
        xi = X_m[:,i]; Xo = np.delete(X_m,i,axis=1)
        Xo = np.hstack([np.ones((len(xi),1)),Xo])
        b  = np.linalg.pinv(Xo.T@Xo) @ Xo.T @ xi
        r2i = 1 - np.sum((xi-Xo@b)**2)/np.sum((xi-xi.mean())**2)
        vif = 1/(1-r2i) if r2i<1 else 9999
        rows.append({'Variable':var,'VIF':round(vif,2),
                     'Corr_with_Dep':round(np.corrcoef(xi,y_dep)[0,1],4),
                     'Level':('SEVERE' if vif>10 else 'Moderate' if vif>5 else 'Low')})
    return pd.DataFrame(rows)

vif_df = compute_vif(X_raw, independent_vars, y)
print(vif_df.to_string(index=False))

## 4. OLS — Unconstrained Reference

In [ ]:
XtX_inv = np.linalg.pinv(X_orig.T @ X_orig)
b_ols   = XtX_inv @ X_orig.T @ y
yp_ols  = X_orig @ b_ols
ss_tot  = np.sum((y-y.mean())**2)
r2_ols  = 1 - np.sum((y-yp_ols)**2)/ss_tot
adj_r2_ols = 1-(1-r2_ols)*(n-1)/(n-p-1)
mape_ols   = mean_absolute_percentage_error(y,yp_ols)*100
res_ols    = y-yp_ols
dw_ols     = np.sum(np.diff(res_ols)**2)/np.sum(res_ols**2)
se_ols     = np.sqrt(np.diag(np.sum(res_ols**2)/(n-p-1)*XtX_inv))
t_ols      = b_ols/se_ols
pv_ols     = 2*stats.t.sf(np.abs(t_ols),df=n-p-1)

print(f'OLS: R2={r2_ols:.4f} | Adj-R2={adj_r2_ols:.4f} | MAPE={mape_ols:.2f}% | DW={dw_ols:.4f}')
ols_df = pd.DataFrame({'Variable':cols,'Coef':np.round(b_ols,3),
    'SE':np.round(se_ols,3),'t':np.round(t_ols,3),'p':np.round(pv_ols,4),
    'Sig':['***' if p<0.01 else '**' if p<0.05 else '*' if p<0.10 else '' for p in pv_ols]})
print(ols_df.to_string(index=False))

## 5. Constrained MLR Setup

### Recommended Contribution Bounds (Revised from Original)

| Variable | Sign | Min % | Max % | Change from Original |
|---|---|---|---|---|
| TV | + | 2% | 12% | Slightly widened |
| YouTube | + | 1% | 8% | Widened |
| Radio | + | 1% | 8% | Widened |
| Print | + | 3% | 16% | Widened |
| Gas Price | − | −12% | −0.5% | Changed from −20/−10 |
| **Discount** | **+** | **45%** | **72%** | **↑ Revised up from 30–40%** |
| **Seasonality** | **+** | **45%** | **72%** | **↑ Revised up from 15–20%** |

In [ ]:
X_scale = X_raw.std(axis=0); X_scale[X_scale==0]=1.0
y_scale = y.std()
X_sc    = np.hstack([np.ones((n,1)), X_raw/X_scale])
y_sc    = y/y_scale
total_y = y.sum()

CONTRIB_BOUNDS = {
    'TVT_SUP': ( 0.020,  0.120),
    'YTB_SUP': ( 0.010,  0.080),
    'RAD_SUP': ( 0.010,  0.080),
    'PRT_SUP': ( 0.030,  0.160),
    'GAS_PRI': (-0.120, -0.005),
    'DIS_COU': ( 0.450,  0.720),
    'SEA_ALT': ( 0.450,  0.720),
}

eps=1e-6
SIGNS={'TVT_SUP':'pos','YTB_SUP':'pos','RAD_SUP':'pos','PRT_SUP':'pos',
        'GAS_PRI':'neg','DIS_COU':'pos','SEA_ALT':'pos'}
opt_bounds = [(None,None)]+[(eps,None) if SIGNS.get(c)=='pos' else
              (None,-eps) if SIGNS.get(c)=='neg' else (None,None)
              for c in independent_vars]

MAT_MASKS  = {m:(data['MAT']==m).values for m in ['MAT_2018','MAT_2019','MAT_2020']}
MAT_TOTALS = {m:y[mask].sum() for m,mask in MAT_MASKS.items()}

constraints=[]
for mat,mask in MAT_MASKS.items():
    Xm=X_sc[mask]; tot=MAT_TOTALS[mat]
    for var,(mn,mx) in CONTRIB_BOUNDS.items():
        idx=cols.index(var)
        constraints.append({'type':'ineq','fun':
            lambda b,i=idx,Xm_=Xm,t=tot,m=mn: (np.sum(b[i]*Xm_[:,i])*y_scale)/t-m})
        constraints.append({'type':'ineq','fun':
            lambda b,i=idx,Xm_=Xm,t=tot,m=mx: m-(np.sum(b[i]*Xm_[:,i])*y_scale)/t})
print(f'Constraints: {len(constraints)}')

## 6. Run Optimisation (20-start SLSQP)

In [ ]:
b_init=np.zeros(p+1); b_init[0]=0.25*total_y/(n*y_scale)
for var,(mn,mx) in CONTRIB_BOUNDS.items():
    idx=cols.index(var); sx=np.sum(X_sc[:,idx])
    if abs(sx)>1e-10: b_init[idx]=(mn+mx)/2*total_y/(sx*y_scale)

np.random.seed(42)
best_r2,best_b,best_res=-999,None,None
for k in range(20):
    b0=b_init+np.random.randn(p+1)*(0.02 if k<10 else 0.08)
    for i,(lo,hi) in enumerate(opt_bounds):
        if lo: b0[i]=max(b0[i],lo)
        if hi: b0[i]=min(b0[i],hi)
    res=minimize(lambda b:np.sum((y_sc-X_sc@b)**2),b0,method='SLSQP',
                 bounds=opt_bounds,constraints=constraints,
                 options={'maxiter':500000,'ftol':1e-15,'eps':1e-9})
    b=res.x
    bo=np.array([b[0]*y_scale]+[b[i+1]*y_scale/X_scale[i] for i in range(p)])
    r2t=1-np.sum((y-X_orig@bo)**2)/ss_tot
    if r2t>best_r2: best_r2,best_b,best_res=r2t,bo,res

b_con=best_b; yp_c=X_orig@b_con; rc=y-yp_c
r2_c   = 1-np.sum(rc**2)/ss_tot
adj_r2c= 1-(1-r2_c)*(n-1)/(n-p-1)
mape_c = mean_absolute_percentage_error(y,yp_c)*100
dw_c   = np.sum(np.diff(rc)**2)/np.sum(rc**2)
se_c   = np.sqrt(np.diag(np.sum(rc**2)/(n-p-1)*XtX_inv))
t_c    = b_con/se_c; pv_c=2*stats.t.sf(np.abs(t_c),df=n-p-1)
print(f'Constrained: R2={r2_c:.4f} | AdjR2={adj_r2c:.4f} | MAPE={mape_c:.2f}% | DW={dw_c:.4f}')
con_df=pd.DataFrame({'Variable':cols,'Coef':np.round(b_con,3),'t':np.round(t_c,3),
    'p':np.round(pv_c,4),'Sig':['***' if p<0.01 else '**' if p<0.05 else '*' if p<0.10 else '' for p in pv_c]})
print(con_df.to_string(index=False))

## 7. Contribution by MAT Period

In [ ]:
for mat in ['MAT_2018','MAT_2019','MAT_2020']:
    mask=MAT_MASKS[mat]; Xm=X_orig[mask]; ym=y[mask]; tot=ym.sum()
    print(f'\n{mat}  Actual={tot:,.0f}')
    for i,var in enumerate(cols):
        c=np.sum(b_con[i]*Xm[:,i]); sh=c/tot*100
        bn=CONTRIB_BOUNDS.get(var)
        ok='✓' if not bn else ('✓' if bn[0]*100-0.1<=sh<=bn[1]*100+0.1 else '✗')
        bstr=f'{bn[0]*100:.0f}%-{bn[1]*100:.0f}%' if bn else 'free'
        print(f'  {var:<12}: {c:>10,.1f}  ({sh:>6.2f}%)  [{bstr}]  {ok}')

## 8. Run Full Export (calls Altima_MMM_v2.py)

The script `Altima_MMM_v2.py` generates `MMM_Model_Output_v2.xlsx` with 8 tabs:
- **Model_Metrics** — OLS vs Constrained performance comparison
- **OLS_Unconstrained** — Full coefficient table with significance
- **Constrained_MLR** — Sign-compliant coefficient table
- **Contribution_MAT** — Per-MAT decomposition with bounds check
- **VIF_Multicollinearity** — VIF table with action flags
- **Weekly_Volume** — Weekly decomposition by variable
- **MAT_Summary** — Actual vs Predicted by MAT year
- **Recommended_Bounds** — Revised bounds with rationale

In [ ]:
# Run the full export
import subprocess
result = subprocess.run(['python3','Altima_MMM_v2.py'], capture_output=True, text=True)
print(result.stdout[-2000:] if len(result.stdout)>2000 else result.stdout)
if result.returncode==0: print('\nMMM_Model_Output_v2.xlsx generated successfully.')
else: print('ERROR:', result.stderr[-1000:])